# PWKD Option 3 — Prune ResNet18 Directly

Implements **Pruning While Knowledge Distillation** (Wang et al., 2025) using:
- **Teacher**: frozen pretrained ResNet30 (RadImageNet weights)
- **Student**: a deep copy of the *same* pretrained ResNet30, fine-tuned
  with PWKD simultaneously pruning and distilling

This is a same-architecture setup, closely analogous to the paper's
EDSR-32-256 → EDSR-16-64 setup but keeping the architecture fixed and
instead driving channels to zero through the differentiable sparsity penalty.
The wavelet channel-projection step is skipped (teacher/student channels match).

Starting from pretrained weights means the student begins at the teacher's
F1 level (~0.4) and PWKD nudges it toward a smaller, slightly lower-F1 model —
a much more favourable trade-off than training from scratch.

Produces 5 compressed models at pruning ratios [10%, 25%, 50%, 70%, 90%],
saved to `trained_models/pwkd_self_r18_<ratio>/` and uploaded to HuggingFace.

**Run all cells top to bottom. Requires GPU.**

In [1]:
import os, copy, subprocess
import torch

target = 'CS6423_knowledge_distillation_project'
if not os.getcwd().endswith(target):
    import sys
    os.chdir(os.path.join(os.getcwd(), target))
    if os.getcwd() not in sys.path:
        sys.path.insert(0, os.getcwd())

print(f'Working dir: {os.getcwd()}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

subprocess.run(['pip', 'install', 'PyWavelets', '--quiet'], check=True)

Working dir: /home/cor10/CS6423_knowledge_distillation_project
Device: cuda


CompletedProcess(args=['pip', 'install', 'PyWavelets', '--quiet'], returncode=0)

In [2]:
import pandas as pd
from modules.dataset_prepper import datasetPrepper

data_prep = datasetPrepper(
    dataframe_path='data/labels.csv',
    image_dir='data/test_images',
).prepare(compute_class_weights=True)

NUM_CLASSES = len(data_prep.class_names)
print(f'Classes: {NUM_CLASSES}')
print(f'Train batches: {len(data_prep.train_loader)} | Val batches: {len(data_prep.val_loader)}')

Classes: 61
Train batches: 249 | Val batches: 50


In [3]:
from modules.imagenet_loader import ImagenetLoader
from modules.evaluate_model import ModelEvaluator

loader = ImagenetLoader()

# Load pretrained ResNet18 — this serves as both the frozen teacher
# and the starting point for each student deep copy
resnet18_pretrained = loader.load_radimagenet_resnet18(
    weights_path='trained_models/resnet18_baseline_gpu_new/resnet18_baseline_gpu_new.pth',
    load_type='load'
)
resnet18_pretrained = resnet18_pretrained.to(device)

evaluator = ModelEvaluator(
    data_loader=data_prep.val_loader,
    class_names=data_prep.class_names,
    device=str(device),
)

baseline_metrics = evaluator.evaluate_single(resnet18_pretrained, 'ResNet18_baseline')
BASELINE_PARAMS  = baseline_metrics['total_parameters']
print(f'ResNet18 baseline F1:    {baseline_metrics["f1_macro"]:.4f}')
print(f'ResNet18 baseline params: {BASELINE_PARAMS:,}')


Warming up ResNet18_baseline...
Running inference...
ResNet18 baseline F1:    0.4054
ResNet18 baseline params: 11,207,805


In [4]:
import torch.nn as nn
import copy
from modules.model_trainer import modelTrainer
from modules.evaluate_model import ModelEvaluator
from pwkd import PWKDLoss, make_aux_fn, finalise_student

RESNET18_CHANNELS = {'layer2': 128, 'layer3': 256, 'layer4': 512}

PRUNING_RATIOS = [0.10, 0.25, 0.50, 0.70, 0.90]
NUM_EPOCHS     = 10
LAM            = 0.2
KD_TEMP        = 4.0
SPARSE_WEIGHT  = 1e-4

evaluator = ModelEvaluator(
    data_loader=data_prep.val_loader,
    class_names=data_prep.class_names,
    device=str(device),
)

teacher = copy.deepcopy(resnet18_pretrained).eval()
for p in teacher.parameters():
    p.requires_grad = False

pwkd_metrics = []

print(f'ResNet18 baseline F1:    {baseline_metrics["f1_macro"]:.4f}')
print(f'ResNet18 baseline params: {BASELINE_PARAMS:,}')


for ratio in PRUNING_RATIOS:
    label = f'pwkd_self_r18_r{int(ratio * 100)}'
    print(f'\n{"="*60}')
    print(f'  PWKD Option 3 (ResNet18) — pruning ratio {ratio:.0%}')
    print(f'{"="*60}')

    student = copy.deepcopy(resnet18_pretrained).to(device)
    for p in student.parameters():
        p.requires_grad = True

    pwkd_loss = PWKDLoss(
        student          = student,
        teacher          = teacher,
        pruning_ratio    = ratio,
        teacher_channels = RESNET18_CHANNELS,
        student_channels = RESNET18_CHANNELS,
        class_weights    = data_prep.class_weights.to(device)
                           if data_prep.class_weights is not None else None,
        lam              = LAM,
        kd_temp          = KD_TEMP,
        sparse_weight    = SPARSE_WEIGHT,
    ).to(device)

    trainer = modelTrainer(
        model      = student,
        data_prep  = data_prep,
        device     = device,
        learn_rate = 2e-4,
        num_epochs = NUM_EPOCHS,
        model_name = label,
    )
    trainer.loss_fn   = pwkd_loss
    trainer.optimizer = torch.optim.AdamW(
        list(student.parameters()) + list(pwkd_loss.parameters()),
        lr=2e-4, weight_decay=1e-2,
    )
    trainer.create_classnum_to_label_map(data_prep.class_names)
    trainer.train_all(save_as_object=True, aux_forward_fn=make_aux_fn(teacher), pwkd=True)

    student = finalise_student(student, pwkd_loss, data_prep.train_loader, device)

    import os
    save_dir = os.path.join('trained_models', label)
    os.makedirs(save_dir, exist_ok=True)
    torch.save({'model': student, 'epoch': NUM_EPOCHS},
               os.path.join(save_dir, f'{label}_full.pth'))
    print(f'Saved → {save_dir}/{label}_full.pth')

    student.eval()
    metrics = evaluator.evaluate_single(student, label)

    pwkd_metrics.append({
        'Pruning Ratio':      f'{int(ratio*100)}%',
        'Size Reduction (%)': round((BASELINE_PARAMS - metrics['total_parameters'])
                                    / BASELINE_PARAMS * 100, 2),
        'F1 Score':           round(metrics['f1_macro'],     4),
        'Size (MB)':          round(metrics['model_size_mb'], 1),
        'Latency (ms)':       round(metrics['avg_latency_ms'], 2),
    })
    print(f'  ratio: {ratio:.0%} | F1: {metrics["f1_macro"]:.4f} | '
          f'params: {metrics["total_parameters"]:,} | '
          f'latency: {metrics["avg_latency_ms"]:.2f}ms')

summary_df = pd.DataFrame(pwkd_metrics).set_index('Pruning Ratio')
print('\nPWKD Option 3 (ResNet18) — Results')
display(summary_df)


ResNet18 baseline F1:    0.4054
ResNet18 baseline params: 11,207,805

  PWKD Option 3 (ResNet18) — pruning ratio 10%
  Epoch 1/10 [  4.8%]  loss: 1.2029
  Epoch 1/10 [  9.6%]  loss: 1.2126
  Epoch 1/10 [ 14.5%]  loss: 1.3565
  Epoch 1/10 [ 19.3%]  loss: 1.3716
  Epoch 1/10 [ 24.1%]  loss: 1.0862
  Epoch 1/10 [ 28.9%]  loss: 1.1640
  Epoch 1/10 [ 33.7%]  loss: 0.9825
  Epoch 1/10 [ 38.6%]  loss: 0.8092
  Epoch 1/10 [ 43.4%]  loss: 0.9097
  Epoch 1/10 [ 48.2%]  loss: 0.8727
  Epoch 1/10 [ 53.0%]  loss: 0.8067
  Epoch 1/10 [ 57.8%]  loss: 0.7593
  Epoch 1/10 [ 62.7%]  loss: 0.7108
  Epoch 1/10 [ 67.5%]  loss: 0.7688
  Epoch 1/10 [ 72.3%]  loss: 0.6361
  Epoch 1/10 [ 77.1%]  loss: 0.6338
  Epoch 1/10 [ 81.9%]  loss: 0.6448
  Epoch 1/10 [ 86.7%]  loss: 0.6609
  Epoch 1/10 [ 91.6%]  loss: 0.6519
  Epoch 1/10 [ 96.4%]  loss: 0.5818
  Epoch 1/10 [100.0%]  loss: 0.6252


Validating: 100%|██████████| 50/50 [00:02<00:00, 18.20batch/s]


Epoch 1/10
Train Loss: 0.8815 | Train F1: 0.5909
Val Loss: 1.6380 | Val F1: 0.3563
Epoch Time: 34.66s



  Epoch 2/10 [  4.8%]  loss: 0.6096
  Epoch 2/10 [  9.6%]  loss: 0.6121
  Epoch 2/10 [ 14.5%]  loss: 0.5500
  Epoch 2/10 [ 19.3%]  loss: 0.5767
  Epoch 2/10 [ 24.1%]  loss: 0.5175
  Epoch 2/10 [ 28.9%]  loss: 0.5362
  Epoch 2/10 [ 33.7%]  loss: 0.5298
  Epoch 2/10 [ 38.6%]  loss: 0.5356
  Epoch 2/10 [ 43.4%]  loss: 0.5539
  Epoch 2/10 [ 48.2%]  loss: 0.5138
  Epoch 2/10 [ 53.0%]  loss: 0.4959
  Epoch 2/10 [ 57.8%]  loss: 0.5325
  Epoch 2/10 [ 62.7%]  loss: 0.4847
  Epoch 2/10 [ 67.5%]  loss: 0.4380
  Epoch 2/10 [ 72.3%]  loss: 0.4894
  Epoch 2/10 [ 77.1%]  loss: 0.4471
  Epoch 2/10 [ 81.9%]  loss: 0.4803
  Epoch 2/10 [ 86.7%]  loss: 0.4789
  Epoch 2/10 [ 91.6%]  loss: 0.4255
  Epoch 2/10 [ 96.4%]  loss: 0.4202
  Epoch 2/10 [100.0%]  loss: 0.4859


Validating: 100%|██████████| 50/50 [00:03<00:00, 16.38batch/s]


...

  Epoch 10/10 [  4.8%]  loss: 0.3235
  Epoch 10/10 [  9.6%]  loss: 0.3150
  Epoch 10/10 [ 14.5%]  loss: 0.3582
  Epoch 10/10 [ 19.3%]  loss: 0.3457
  Epoch 10/10 [ 24.1%]  loss: 0.3146
  Epoch 10/10 [ 28.9%]  loss: 0.3454
  Epoch 10/10 [ 33.7%]  loss: 0.3270
  Epoch 10/10 [ 38.6%]  loss: 0.3254
  Epoch 10/10 [ 43.4%]  loss: 0.3252
  Epoch 10/10 [ 48.2%]  loss: 0.3492
  Epoch 10/10 [ 53.0%]  loss: 0.3455
  Epoch 10/10 [ 57.8%]  loss: 0.3350
  Epoch 10/10 [ 62.7%]  loss: 0.3477
  Epoch 10/10 [ 67.5%]  loss: 0.3276
  Epoch 10/10 [ 72.3%]  loss: 0.3269
  Epoch 10/10 [ 77.1%]  loss: 0.3412
  Epoch 10/10 [ 81.9%]  loss: 0.3366
  Epoch 10/10 [ 86.7%]  loss: 0.3241
  Epoch 10/10 [ 91.6%]  loss: 0.3297
  Epoch 10/10 [ 96.4%]  loss: 0.3266
  Epoch 10/10 [100.0%]  loss: 0.3527


Validating: 100%|██████████| 50/50 [00:02<00:00, 17.95batch/s]


Epoch 10/10
Train Loss: 0.3342 | Train F1: 0.8363
Val Loss: 1.4871 | Val F1: 0.4209
Epoch Time: 35.16s



Finalised: 1724/1920 conv1 channels zeroed (89.8%)
Saved → trained_models/pwkd_self_r18_r90/pwkd_self_r18_r90_full.pth

Warming up pwkd_self_r18_r90...
Running inference...
  ratio: 90% | F1: 0.0002 | params: 1,339,197 | latency: 0.39ms

PWKD Option 3 (ResNet18) — Results


,Size Reduction (%),F1 Score,Size (MB),Latency (ms)
Pruning Ratio,,,,
10%,9.69,0.3858,42.8,0.39
25%,24.50,0.2268,42.8,0.38
50%,49.01,0.0008,42.8,0.39
70%,68.50,0.0002,42.8,0.38
90%,88.05,0.0002,42.8,0.39


In [6]:
# ── Extra ratios to better capture the Pareto curve descent ──────────────────
# Run this cell independently — existing trained models are not affected.

EXTRA_RATIOS = [0.2, 0.3, 0.35, 0.40, 0.45]   # between the good and collapsed points

for ratio in EXTRA_RATIOS:
    label = f'pwkd_self_r18_r{int(ratio * 100)}'
    print(f'\n{"="*60}')
    print(f'  PWKD Option 3 (ResNet18) — pruning ratio {ratio:.0%}')
    print(f'{"="*60}')

    student = copy.deepcopy(resnet18_pretrained).to(device)
    for p in student.parameters():
        p.requires_grad = True

    pwkd_loss = PWKDLoss(
        student          = student,
        teacher          = teacher,
        pruning_ratio    = ratio,
        teacher_channels = RESNET18_CHANNELS,
        student_channels = RESNET18_CHANNELS,
        class_weights    = data_prep.class_weights.to(device)
                           if data_prep.class_weights is not None else None,
        lam              = LAM,
        kd_temp          = KD_TEMP,
        sparse_weight    = SPARSE_WEIGHT,
    ).to(device)

    trainer = modelTrainer(
        model      = student,
        data_prep  = data_prep,
        device     = device,
        learn_rate = 5e-4,
        num_epochs = NUM_EPOCHS,
        model_name = label,
    )
    trainer.loss_fn   = pwkd_loss
    trainer.optimizer = torch.optim.AdamW(
        list(student.parameters()) + list(pwkd_loss.parameters()),
        lr=5e-4, weight_decay=1e-2,
    )
    trainer.create_classnum_to_label_map(data_prep.class_names)
    trainer.train_all(save_as_object=True, aux_forward_fn=make_aux_fn(teacher), pwkd=True)

    student = finalise_student(student, pwkd_loss, data_prep.train_loader, device)

    save_dir = os.path.join('trained_models', label)
    os.makedirs(save_dir, exist_ok=True)
    torch.save({'model': student, 'epoch': NUM_EPOCHS},
               os.path.join(save_dir, f'{label}_full.pth'))
    print(f'Saved → {save_dir}/{label}_full.pth')

    student.eval()
    metrics = evaluator.evaluate_single(student, label)

    pwkd_metrics.append({
        'Pruning Ratio':      f'{int(ratio*100)}%',
        'Size Reduction (%)': round((BASELINE_PARAMS - metrics['total_parameters'])
                                    / BASELINE_PARAMS * 100, 2),
        'F1 Score':           round(metrics['f1_macro'],     4),
        'Size (MB)':          round(metrics['model_size_mb'], 1),
        'Latency (ms)':       round(metrics['avg_latency_ms'], 2),
    })
    print(f'  ratio: {ratio:.0%} | F1: {metrics["f1_macro"]:.4f} | '
          f'params: {metrics["total_parameters"]:,}')

summary_df = pd.DataFrame(pwkd_metrics).set_index('Pruning Ratio')
display(summary_df)


  PWKD Option 3 (ResNet18) — pruning ratio 20%
  Epoch 1/10 [  4.8%]  loss: 1.9581
  Epoch 1/10 [  9.6%]  loss: 2.3039
  Epoch 1/10 [ 14.5%]  loss: 2.3477
  Epoch 1/10 [ 19.3%]  loss: 2.0448
  Epoch 1/10 [ 24.1%]  loss: 1.7762
  Epoch 1/10 [ 28.9%]  loss: 1.7435
  Epoch 1/10 [ 33.7%]  loss: 1.5967
  Epoch 1/10 [ 38.6%]  loss: 1.5793
  Epoch 1/10 [ 43.4%]  loss: 1.4946
  Epoch 1/10 [ 48.2%]  loss: 1.5867
  Epoch 1/10 [ 53.0%]  loss: 1.2651
  Epoch 1/10 [ 57.8%]  loss: 1.2774
  Epoch 1/10 [ 62.7%]  loss: 1.1060
  Epoch 1/10 [ 67.5%]  loss: 1.2242
  Epoch 1/10 [ 72.3%]  loss: 1.0655
  Epoch 1/10 [ 77.1%]  loss: 1.0708
  Epoch 1/10 [ 81.9%]  loss: 1.0910
  Epoch 1/10 [ 86.7%]  loss: 1.0435
  Epoch 1/10 [ 91.6%]  loss: 1.0059
  Epoch 1/10 [ 96.4%]  loss: 1.0149
  Epoch 1/10 [100.0%]  loss: 0.9300


Validating: 100%|██████████| 50/50 [00:02<00:00, 18.14batch/s]


Epoch 1/10
Train Loss: 1.4599 | Train F1: 0.4103
Val Loss: 2.1841 | Val F1: 0.2409
Epoch Time: 34.05s



  Epoch 2/10 [  4.8%]  loss: 1.0035
  Epoch 2/10 [  9.6%]  loss: 0.9506
  Epoch 2/10 [ 14.5%]  loss: 0.8132
  Epoch 2/10 [ 19.3%]  loss: 0.9065
  Epoch 2/10 [ 24.1%]  loss: 0.9335
  Epoch 2/10 [ 28.9%]  loss: 0.7876
  Epoch 2/10 [ 33.7%]  loss: 0.8188
  Epoch 2/10 [ 38.6%]  loss: 0.8541
  Epoch 2/10 [ 43.4%]  loss: 0.7839
  Epoch 2/10 [ 48.2%]  loss: 0.7512
  Epoch 2/10 [ 53.0%]  loss: 0.7734
  Epoch 2/10 [ 57.8%]  loss: 0.7353
  Epoch 2/10 [ 62.7%]  loss: 0.6897
  Epoch 2/10 [ 67.5%]  loss: 0.6670
  Epoch 2/10 [ 72.3%]  loss: 0.7133
  Epoch 2/10 [ 77.1%]  loss: 0.6318
  Epoch 2/10 [ 81.9%]  loss: 0.6125
  Epoch 2/10 [ 86.7%]  loss: 0.6754
  Epoch 2/10 [ 91.6%]  loss: 0.6592
  Epoch 2/10 [ 96.4%]  loss: 0.6445
  Epoch 2/10 [100.0%]  loss: 0.6382


Validating: 100%|██████████| 50/50 [00:02<00:00, 17.82batch/s]

...

  Epoch 10/10 [  4.8%]  loss: 0.3807
  Epoch 10/10 [  9.6%]  loss: 0.4024
  Epoch 10/10 [ 14.5%]  loss: 0.4455
  Epoch 10/10 [ 19.3%]  loss: 0.3980
  Epoch 10/10 [ 24.1%]  loss: 0.3986
  Epoch 10/10 [ 28.9%]  loss: 0.4186
  Epoch 10/10 [ 33.7%]  loss: 0.5213
  Epoch 10/10 [ 38.6%]  loss: 0.5893
  Epoch 10/10 [ 43.4%]  loss: 0.5527
  Epoch 10/10 [ 48.2%]  loss: 0.5453
  Epoch 10/10 [ 53.0%]  loss: 0.6273
  Epoch 10/10 [ 57.8%]  loss: 0.6222
  Epoch 10/10 [ 62.7%]  loss: 0.5714
  Epoch 10/10 [ 67.5%]  loss: 0.5427
  Epoch 10/10 [ 72.3%]  loss: 0.5240
  Epoch 10/10 [ 77.1%]  loss: 0.5296
  Epoch 10/10 [ 81.9%]  loss: 0.5124
  Epoch 10/10 [ 86.7%]  loss: 0.5082
  Epoch 10/10 [ 91.6%]  loss: 0.5015
  Epoch 10/10 [ 96.4%]  loss: 0.4950
  Epoch 10/10 [100.0%]  loss: 0.4944


Validating: 100%|██████████| 50/50 [00:02<00:00, 18.16batch/s]


Epoch 10/10
Train Loss: 0.5040 | Train F1: 0.7342
Val Loss: 1.9486 | Val F1: 0.2968
Epoch Time: 35.19s



Finalised: 860/1920 conv1 channels zeroed (44.8%)
Saved → trained_models/pwkd_self_r18_r45/pwkd_self_r18_r45_full.pth

Warming up pwkd_self_r18_r45...
Running inference...
  ratio: 45% | F1: 0.0056 | params: 6,276,669


,Size Reduction (%),F1 Score,Size (MB),Latency (ms)
Pruning Ratio,,,,
10%,9.69,0.3858,42.8,0.39
25%,24.50,0.2268,42.8,0.38
50%,49.01,0.0008,42.8,0.39
70%,68.50,0.0002,42.8,0.38
90%,88.05,0.0002,42.8,0.39
20%,19.49,0.3177,42.8,0.38
30%,29.24,0.1643,42.8,0.39
35%,34.20,0.0786,42.8,0.39
40%,39.04,0.0138,42.8,0.38
